In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

class FivePhaseModel:
    """
    Implementa un modello di nanoindentazione a 5 fasi.
    Le equazioni descrivono la relazione tra forza (F) e spostamento (h)
    in diverse fasi di carico e scarico.
    - Fase 1 (Loading): F = a1 * (h - h1)^m1
    - Fase 3 (Unloading): F = a2 * (h - h2)^m2
    - Fase 5 (Unload Light): F = a3 * (h - h3)^m3
    Le fasi 2 e 4 sono fasi di mantenimento (hold) a forza costante.
    I parametri h2 e h3 vengono calcolati dinamicamente per garantire la continuità
    dello spostamento tra le fasi.
    """
    def __init__(self, a1, a2, a3, h1, m1, m2, m3):
        self.a1, self.a2, self.a3 = a1, a2, a3
        self.h1 = h1  # Offset di spostamento iniziale (tipicamente 0)
        self.m1, self.m2, self.m3 = m1, m2, m3
        
    def generate_force_profile(self, loading_time, hold1_time, unloading_time, hold2_time, unload_light_time, loading_rate, unloading_rate):
        """
        Genera il profilo di forza nel tempo basandosi sui tempi e le velocità definite.
        """
        dt = 0.01 # Timestep per la simulazione
        
        # Calcola le forze massime e intermedie basandosi sui tempi e le velocità
        F_max = loading_rate * loading_time
        F_intermediate = F_max - (unloading_rate * unloading_time)
        F_intermediate = max(0, F_intermediate) # Assicura che non sia negativa
        
        phase_times = {
            'loading': loading_time,
            'hold1': hold1_time,
            'unloading': unloading_time,
            'hold2': hold2_time,
            'unload_light': unload_light_time
        }
        
        total_time = sum(phase_times.values())
        time_array = np.arange(0, total_time + dt, dt)
        force_array = np.zeros_like(time_array)
        phase_array = np.zeros_like(time_array, dtype=int)
        
        # Tempi cumulativi per le transizioni di fase
        t1 = phase_times['loading']
        t2 = t1 + phase_times['hold1'] 
        t3 = t2 + phase_times['unloading']
        t4 = t3 + phase_times['hold2']
        
        for i, t in enumerate(time_array):
            if t <= t1:  # Fase 1: Loading
                phase_array[i] = 1
                force_array[i] = loading_rate * t
            
            elif t <= t2:  # Fase 2: Hold1 - forza costante
                phase_array[i] = 2
                force_array[i] = F_max
                
            elif t <= t3:  # Fase 3: Unloading
                phase_array[i] = 3
                t_rel = t - t2
                force_array[i] = F_max - unloading_rate * t_rel
                
            elif t <= t4:  # Fase 4: Hold2 - forza costante
                phase_array[i] = 4
                force_array[i] = F_intermediate
                
            else:  # Fase 5: Unload_light - forza va a zero
                phase_array[i] = 5
                t_rel = t - t4
                t_unload_final = phase_times['unload_light']
                if t_unload_final > 0 and F_intermediate > 0:
                    force_array[i] = F_intermediate * (1 - t_rel / t_unload_final)
                    force_array[i] = max(0, force_array[i]) # Assicura che la forza non diventi negativa
                else:
                    force_array[i] = 0
                    
        return time_array, force_array, phase_array
    
    def calculate_displacement_from_force(self, force_array, phase_array):
        """
        Calcola lo spostamento (h) dal profilo di forza, garantendo la continuità.
        """
        h_array = np.zeros_like(force_array)
        
        # Variabili per gestire la continuità tra le fasi
        h_end_loading = 0; h2 = 0; h_end_unloading = 0; h3 = 0
        
        for i, (F, segment) in enumerate(zip(force_array, phase_array)):
            
            if segment == 1 or segment == 0:  # Loading
                if F > 0: h_array[i] = (F / self.a1)**(1 / self.m1) + self.h1
                else: h_array[i] = self.h1
            
            elif segment == 2:  # Hold1
                if i > 0 and phase_array[i-1] == 1: h_end_loading = h_array[i-1]
                h_array[i] = h_end_loading
                
            elif segment == 3:  # Unloading
                if i > 0 and phase_array[i-1] == 2:
                    F_max = force_array[i-1]
                    if F_max > 0: h2 = h_end_loading - (F_max / self.a2)**(1 / self.m2)

                if F > 0: h_array[i] = (F / self.a2)**(1 / self.m2) + h2
                else: h_array[i] = h2
                    
            elif segment == 4:  # Hold2
                if i > 0 and phase_array[i-1] == 3: h_end_unloading = h_array[i-1]
                h_array[i] = h_end_unloading
                
            elif segment == 5 or segment == 6:  # Unload_light
                if i > 0 and phase_array[i-1] == 4:
                    F_intermediate = force_array[i-1]
                    if F_intermediate > 0: h3 = h_end_unloading - (F_intermediate / self.a3)**(1 / self.m3)
                
                if F > 0: h_array[i] = (F / self.a3)**(1 / self.m3) + h3
                else: h_array[i] = h3
            
            h_array[i] = max(0, h_array[i])
            
        return h_array

    def simulate_force_displacement_curve(self, sim_params):
        """ Esegue la simulazione completa. """
        time_array, force_array, phase_array = self.generate_force_profile(
            loading_time=sim_params['loading_time'], hold1_time=sim_params['hold1_time'],
            unloading_time=sim_params['unloading_time'], hold2_time=sim_params['hold2_time'],
            unload_light_time=sim_params['unload_light_time'], loading_rate=sim_params['loading_rate'],
            unloading_rate=sim_params['unloading_rate']
        )
        h_array = self.calculate_displacement_from_force(force_array, phase_array)
        
        return {'time': time_array, 'force': force_array, 'displacement': h_array, 'phase': phase_array}

def get_user_input():
    """ Funzione per raccogliere i parametri della simulazione dall'utente, incluso il percorso di salvataggio. """
    params = {}
    print("--- IMPOSTAZIONE DELLA SIMULAZIONE ---")
    
    while True:
        choice = input("Vuoi usare il modello con controllo termico? (s/n): ").lower()
        if choice in ['s', 'n']:
            params['thermal_control'] = (choice == 's'); break
        print("Input non valido. Per favore, inserisci 's' o 'n'.")

    def get_float_input(prompt, min_val=None):
        while True:
            try:
                value = float(input(prompt))
                if min_val is not None and value < min_val:
                    print(f"Errore: il valore deve essere maggiore o uguale a {min_val}."); continue
                return value
            except ValueError:
                print("Input non valido. Per favore, inserisci un numero.")

    params['loading_rate'] = get_float_input("Inserisci la velocità di carico [mN/s]: ", min_val=1e-6)
    params['unloading_rate'] = get_float_input("Inserisci la velocità di scarico [mN/s]: ", min_val=1e-6)
    print("-" * 20)
    params['loading_time'] = get_float_input("Durata della fase di carico (Loading) [s]: ", min_val=0.1)
    params['hold1_time'] = get_float_input("Durata mantenimento a forza massima (Hold1) [s]: ", min_val=0)
    params['unloading_time'] = get_float_input("Durata dello scarico parziale (Unloading) [s]: ", min_val=0.1)
    params['hold2_time'] = get_float_input("Durata mantenimento a forza intermedia (Hold2) [s]: ", min_val=0)
    params['unload_light_time'] = get_float_input("Durata scarico finale a zero (Unload Light) [s]: ", min_val=0)

    # NUOVO: Richiesta del percorso di salvataggio
    save_path = input("Inserisci il percorso completo per salvare i file (es. C:\\Users\\tuo_utente\\Desktop\\Risultati): ")
    params['save_path'] = save_path
    
    return params

# --- SCRIPT PRINCIPALE ---
if __name__ == "__main__":
    
    # Definisci i due set di parametri del modello
    params_thermal = {
        "a1": 9.65e-05, "m1": 2.0514, "a2": 0.0511,  "m2": 1.0746,
        "a3": 0.0323,  "m3": 1.6057, "h1": 0.0
    }
    params_no_thermal = {
        "a1": 0.0001, "m1": 2.0420, "a2": 0.0247, "m2": 1.2138,
        "a3": 0.0057, "m3": 2.0921, "h1": 0.0
    }
    
    sim_params = get_user_input()
    
    model_params = params_thermal if sim_params['thermal_control'] else params_no_thermal
    model_type_str = "CON" if sim_params['thermal_control'] else "SENZA"
    print(f"\n--> Modello {model_type_str} controllo termico selezionato.")
    
    model = FivePhaseModel(**model_params)
    results = model.simulate_force_displacement_curve(sim_params)
    
    # --- PLOTTING E SALVATAGGIO ---
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
    segment_names = {1: 'Loading', 2: 'Hold 1', 3: 'Unloading', 4: 'Hold 2', 5: 'Unload Light'}
    
    # Usa il percorso inserito dall'utente
    save_path = sim_params['save_path']
    if not os.path.exists(save_path): 
        try:
            os.makedirs(save_path)
            print(f"Cartella di salvataggio creata: {save_path}")
        except OSError as e:
            print(f"Errore nella creazione della cartella: {e}")
            save_path = "." # Fallback alla directory corrente
            print("Salvataggio nella directory corrente.")
    
    # --- 1. Salvataggio Grafico: Forza vs Tempo ---
    fig1, ax1 = plt.subplots(figsize=(8, 6))
    for phase in range(1, 6):
        mask = results['phase'] == phase
        if np.any(mask): ax1.plot(results['time'][mask], results['force'][mask], color=colors[phase-1], linewidth=3, label=segment_names[phase])
    ax1.set_xlabel('Tempo (s)'); ax1.set_ylabel('Forza F (mN)'); ax1.legend(); ax1.grid(True, alpha=0.3); ax1.set_title('Profilo di Forza Controllato')
    plt.tight_layout()
    file_path1 = os.path.join(save_path, '1_forza_vs_tempo.jpg')
    fig1.savefig(file_path1, dpi=300); plt.close(fig1)
    print(f"Grafico 1 salvato in: {file_path1}")

    # --- 2. Salvataggio Grafico: Spostamento vs Tempo ---
    fig2, ax2 = plt.subplots(figsize=(8, 6))
    for phase in range(1, 6):
        mask = results['phase'] == phase
        if np.any(mask): ax2.plot(results['time'][mask], results['displacement'][mask], color=colors[phase-1], linewidth=3, label=segment_names[phase])
    ax2.set_xlabel('Tempo (s)'); ax2.set_ylabel('Spostamento h (nm)'); ax2.legend(); ax2.grid(True, alpha=0.3); ax2.set_title('Risposta in Spostamento')
    plt.tight_layout()
    file_path2 = os.path.join(save_path, '2_spostamento_vs_tempo.jpg')
    fig2.savefig(file_path2, dpi=300); plt.close(fig2)
    print(f"Grafico 2 salvato in: {file_path2}")
    
    # --- 3. Salvataggio Grafico: Curva F vs h ---
    fig3, ax3 = plt.subplots(figsize=(8, 6))
    for phase in range(1, 6):
        mask = results['phase'] == phase
        if np.any(mask): ax3.plot(results['force'][mask], results['displacement'][mask], color=colors[phase-1], linewidth=3, label=segment_names[phase])
    ax3.set_xlabel('Forza F (mN)'); ax3.set_ylabel('Spostamento h (nm)'); ax3.legend(); ax3.grid(True, alpha=0.3); ax3.set_title('Curva Forza-Spostamento')
    plt.tight_layout()
    file_path3 = os.path.join(save_path, '3_forza_vs_spostamento.jpg')
    fig3.savefig(file_path3, dpi=300); plt.close(fig3)
    print(f"Grafico 3 salvato in: {file_path3}")

    # --- 4. NUOVO: SALVATAGGIO DATI IN FILE TXT ---
    file_path_txt = os.path.join(save_path, '4_dati_simulazione.txt')
    header = "Time (s)\tPd (nm)\tFn (mN)\tSegmentID\n"
    
    with open(file_path_txt, 'w') as f:
        f.write(header)
        for i in range(len(results['time'])):
            time_val = results['time'][i]
            disp_val = results['displacement'][i]
            force_val = results['force'][i]
            phase_val = results['phase'][i]
            line = f"{time_val:.4f}\t{disp_val:.4f}\t{force_val:.4f}\t{phase_val}\n"
            f.write(line)
    print(f"Dati salvati in: {file_path_txt}")

    # --- VISUALIZZAZIONE DELLA GRIGLIA COMPLETA A SCHERMO ---
    print("\nVisualizzo la griglia completa dei grafici a schermo...")
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Risultati della Simulazione', fontsize=16)

    for phase in range(1, 6): # Plotting comune
        mask = results['phase'] == phase
        if np.any(mask):
            ax1.plot(results['time'][mask], results['force'][mask], color=colors[phase-1], linewidth=2.5, label=segment_names[phase])
            ax2.plot(results['time'][mask], results['displacement'][mask], color=colors[phase-1], linewidth=2.5, label=segment_names[phase])
            ax3.plot(results['force'][mask], results['displacement'][mask], color=colors[phase-1], linewidth=2.5, label=segment_names[phase])
            
    ax1.set_title('INPUT: Profilo di Forza'); ax1.set_xlabel('Tempo (s)'); ax1.set_ylabel('Forza F (mN)'); ax1.legend(); ax1.grid(True, alpha=0.3)
    ax2.set_title('OUTPUT: Risposta in Spostamento'); ax2.set_xlabel('Tempo (s)'); ax2.set_ylabel('Spostamento h (nm)'); ax2.legend(); ax2.grid(True, alpha=0.3)
    ax3.set_title('RISULTATO: Curva F-h'); ax3.set_xlabel('Forza F (mN)'); ax3.set_ylabel('Spostamento h (nm)'); ax3.legend(); ax3.grid(True, alpha=0.3)
    
    ax4.plot(results['time'], results['displacement'], 'k-', linewidth=2, label='Spostamento')
    transitions = np.where(np.diff(results['phase']))[0] + 1
    ax4.plot(results['time'][transitions], results['displacement'][transitions], 'ro', markersize=6, label='Transizione Fase', linestyle='None')
    ax4.set_title('Verifica Continuità Spostamento'); ax4.set_xlabel('Tempo (s)'); ax4.set_ylabel('Spostamento h (nm)'); ax4.legend(); ax4.grid(True, alpha=0.3)
    
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

    print("\n--- RIEPILOGO SIMULAZIONE ---")
    print(f"- Forza massima calcolata: {np.max(results['force']):.2f} mN")
    print(f"- Spostamento massimo: {np.max(results['displacement']):.3f} nm")
    print(f"- Spostamento residuo finale: {results['displacement'][-1]:.3f} nm")